# Ponimator: Unfolding Interactive Pose for Versatile Human-Human Interaction Animation
## Multi-Environment Demo (Google Colab & Local)

[![arXiv](https://img.shields.io/badge/arXiv-2510.14976-b31b1b.svg)](https://arxiv.org/abs/2510.14976) 
[![Project Page](https://img.shields.io/badge/Project-Website-blue?style=flat&logo=Google%20chrome&logoColor=blue)](https://stevenlsw.github.io/ponimator/)

This notebook demonstrates Ponimator for interactive human-human pose animation and generation from monocular images or video frames.

**Environment Support:**
- ☁️ **Google Colab**: Requires GPU runtime (`Runtime > Change runtime type > GPU`)
- 💻 **Local Ubuntu**: Requires NVIDIA GPU with CUDA support

**Updated:** Now using PyTorch 2.9+ for improved compatibility.

## 1. Check GPU Availability

In [ ]:
import torch
import sys
import os

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
environment = "Google Colab" if IN_COLAB else "Local Machine"

print(f"🖥️  Environment: {environment}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"\n{'='*60}")

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA version: {torch.version.cuda}")
    capability = torch.cuda.get_device_capability(0)
    print(f"   GPU Compute Capability: {capability[0]}.{capability[1]}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: No GPU detected!")
    if IN_COLAB:
        print("\n📝 To enable GPU in Google Colab:")
        print("   1. Go to Runtime > Change runtime type")
        print("   2. Select 'T4 GPU' or 'A100 GPU' from Hardware accelerator")
        print("   3. Click Save and wait for runtime to restart")
    else:
        print("\n📝 For local Ubuntu machine:")
        print("   1. Verify NVIDIA GPU is installed: nvidia-smi")
        print("   2. Check CUDA installation: nvcc --version")

print(f"{'='*60}\n")

## 2. Mount Google Drive (Colab Only)

In [ ]:
# Mount Google Drive (Colab only)
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted at /content/drive")
else:
    print("ℹ️  Skipping Google Drive mount (not in Colab environment)")
    print("   For local usage, use absolute paths to your data files")

## 3. Clone Repository and Install Dependencies

In [ ]:
# Clone the repository (Colab only)
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("☁️  Cloning repository for Colab environment...")
    !git clone https://github.com/stevenlsw/ponimator.git
    %cd ponimator
    print("✅ Repository cloned")
else:
    print("💻 Running on local machine - skipping repository clone")
    print("   Assuming we're already in the ponimator directory")
    print(f"   Current directory: {os.getcwd()}")
    
    # Verify we're in the right directory
    if not os.path.exists('ponimator') or not os.path.exists('scripts'):
        print("⚠️  Warning: Expected files not found. Make sure you're in the ponimator directory.")
    else:
        print("✅ Repository directory verified")

In [ ]:
# Install PyTorch 2.9+ and dependencies
print("📦 Installing PyTorch 2.9+ with CUDA 12.1...")

# Install PyTorch 2.9+
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install other dependencies (including aitviewer for SMPL layer)
!pip install -q yacs roma einops scipy scikit-learn aitviewer

print("✅ Dependencies installed")

# Verify installations
import torch
import numpy as np
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

## 4. Download Model Checkpoints

In [ ]:
# Download checkpoints based on environment
import os
import sys
import shutil

IN_COLAB = 'google.colab' in sys.modules

os.makedirs('checkpoints', exist_ok=True)

checkpoint_files = {
    'contactmotion.ckpt': 'https://huggingface.co/shaoweiliu/ponimator/resolve/main/contactmotion.ckpt',
    'contactpose.ckpt': 'https://huggingface.co/shaoweiliu/ponimator/resolve/main/contactpose.ckpt'
}

for filename, url in checkpoint_files.items():
    checkpoint_path = f'checkpoints/{filename}'
    
    if os.path.exists(checkpoint_path):
        size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
        print(f"✅ {filename} already exists ({size_mb:.1f} MB)")
    else:
        if IN_COLAB:
            # Try copying from Google Drive first
            drive_path = f'/content/drive/MyDrive/ponimator/checkpoints/{filename}'
            if os.path.exists(drive_path):
                print(f"📥 Copying {filename} from Google Drive...")
                shutil.copy2(drive_path, checkpoint_path)
                size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
                print(f"✅ {filename} copied ({size_mb:.1f} MB)")
            else:
                print(f"📥 Downloading {filename} from HuggingFace...")
                !wget -q {url} -O {checkpoint_path}
                if os.path.exists(checkpoint_path):
                    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
                    print(f"✅ {filename} downloaded ({size_mb:.1f} MB)")
        else:
            print(f"📥 Downloading {filename} from HuggingFace...")
            !wget -q {url} -O {checkpoint_path}
            if os.path.exists(checkpoint_path):
                size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
                print(f"✅ {filename} downloaded ({size_mb:.1f} MB)")

print("\n✅ All checkpoints ready!")

## 5. Setup SMPL-X Body Models

In [ ]:
# Setup SMPL-X models based on environment
import os
import sys
import shutil

IN_COLAB = 'google.colab' in sys.modules

os.makedirs('body_models/smplx', exist_ok=True)

required_files = {
    'SMPLX_MALE.npz': 'SMPL-X male model',
    'SMPLX_FEMALE.npz': 'SMPL-X female model',
    'SMPLX_NEUTRAL.npz': 'SMPL-X neutral model',
    'SMPLX_MALE.pkl': 'SMPL-X male model (pkl)',
    'SMPLX_FEMALE.pkl': 'SMPL-X female model (pkl)',
    'SMPLX_NEUTRAL.pkl': 'SMPL-X neutral model (pkl)',
}

if IN_COLAB:
    # Copy from Google Drive
    base_dir = '/content/drive/MyDrive/ponimator/body_models/smplx'
    
    if not os.path.exists(base_dir):
        print(f"❌ Source directory not found: {base_dir}")
        print("   Please make sure SMPL-X models are in your Google Drive at:")
        print("   /content/drive/MyDrive/ponimator/body_models/smplx/")
    else:
        print(f"📁 Copying SMPL-X models from: {base_dir}")
        
        for filename, desc in required_files.items():
            source_path = os.path.join(base_dir, filename)
            dest_path = f'body_models/smplx/{filename}'
            
            if os.path.exists(source_path):
                shutil.copy2(source_path, dest_path)
                size_kb = os.path.getsize(dest_path) / 1024
                print(f"  ✓ Copied {desc}: {filename} ({size_kb:.1f} KB)")
            else:
                print(f"  ✗ Not found: {source_path}")
else:
    # Local machine - check if files exist
    print("🔍 Checking for existing SMPL-X models...")
    
    missing_files = []
    for filename, desc in required_files.items():
        file_path = f'body_models/smplx/{filename}'
        
        if os.path.exists(file_path):
            size_kb = os.path.getsize(file_path) / 1024
            print(f"  ✓ {desc}: {filename} ({size_kb:.1f} KB)")
        else:
            print(f"  ✗ {desc}: {filename} NOT FOUND")
            missing_files.append((filename, desc))
    
    if missing_files:
        print(f"\n⚠️  {len(missing_files)} file(s) missing")
        print("\nTo download missing files:")
        print("  1. Register at https://smpl-x.is.tue.mpg.de")
        print("  2. Download SMPL-X models")
        print("  3. Place files in body_models/smplx/")
    else:
        print("\n✅ All SMPL-X models found!")

## 6. Run Interactive Pose Animation (Buddi Data)

This demo uses pre-estimated interactive poses from Buddi to generate motion sequences.

In [ ]:
# Set data path based on environment
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Google Colab: Use Google Drive path
    data_dir = '/content/drive/MyDrive/ponimator/data/buddi/Couple_6806'
    print("☁️  Running in Google Colab")
else:
    # Local machine: Use local path
    data_dir = 'data/buddi/Couple_6806'
    print("💻 Running on local machine")

print(f"📁 Data directory: {data_dir}")

# Verify the data directory exists
if os.path.exists(data_dir):
    print(f"✅ Data directory found")
else:
    print(f"⚠️  Data directory not found!")
    print(f"   Expected location: {data_dir}")

# Set output directory
output_dir = "outputs/Couple_6806"
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Output directory: {output_dir}")

In [ ]:
# Run pose2motion inference with JSON export (no visualization)
import torch

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✓ GPU Ready: {torch.cuda.get_device_name(0)}")

# Run inference
!python scripts/run_pose2motion.py \
    --data_dir {data_dir} \
    --save_dir {output_dir} \
    --save \
    --disable_vis \
    --export_json

print(f"\n✅ Processing complete! Results saved to: {output_dir}")
print(f"   - motion_pred.pkl: Raw motion data")
print(f"   - poses.json: Structured JSON export")

## 7. Run Interactive Motion Generation (Motion-X Data)

This demo generates interactive motion from single-person poses.

In [ ]:
# Set Motion-X data path based on environment
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Google Colab: Use Google Drive path
    data_dir = '/content/drive/MyDrive/ponimator/data/motionx/Back_Flip_Kungfu_wushu_Trim9_clip1'
    print("☁️  Running in Google Colab")
else:
    # Local machine: Use local path
    data_dir = 'data/motionx/Back_Flip_Kungfu_wushu_Trim9_clip1'
    print("💻 Running on local machine")

print(f"📁 Data directory: {data_dir}")

# Set output directory
output_dir = "outputs/Back_Flip_Kungfu_wushu_Trim9_clip1"
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Output directory: {output_dir}")

# Set generation parameters
text_prompt = "two person hug each other"
seed = 1
inter_time_idx = 15  # Frame index for interactive pose
gender = "male female"  # Gender for two persons

print(f"\n🎯 Generation settings:")
print(f"   Text prompt: {text_prompt}")
print(f"   Seed: {seed}")
print(f"   Interactive frame: {inter_time_idx}")
print(f"   Gender: {gender}")

In [ ]:
# Run single pose to motion generation with JSON export (no visualization)
import torch

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Run inference
!python scripts/run_singlepose2motion.py \
    --data_dir {data_dir} \
    --save_dir {output_dir} \
    --save \
    --disable_vis \
    --export_json \
    --text "{text_prompt}" \
    --seed {seed} \
    --inter_time_idx {inter_time_idx} \
    --gender {gender}

print(f"\n✅ Processing complete! Results saved to: {output_dir}")
print(f"   - motion_pred.pkl: Raw motion data")
print(f"   - poses.json: Structured JSON export")

## 8. Download Results (Colab Only)

In [ ]:
# Download JSON results
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

# Find all poses.json files
poses_files = []
for root, dirs, files in os.walk('outputs'):
    for file in files:
        if file == 'poses.json':
            poses_files.append(os.path.join(root, file))

if IN_COLAB:
    from google.colab import files
    
    for poses_json in poses_files:
        if os.path.exists(poses_json):
            size_mb = os.path.getsize(poses_json) / (1024 * 1024)
            print(f"📥 Downloading: {poses_json} ({size_mb:.2f} MB)")
            files.download(poses_json)
    
    print("\n✅ All JSON files downloaded!")
else:
    print("💻 Running on local machine")
    print("\n📁 JSON files saved at:")
    for poses_json in poses_files:
        if os.path.exists(poses_json):
            size_mb = os.path.getsize(poses_json) / (1024 * 1024)
            print(f"   {os.path.abspath(poses_json)} ({size_mb:.2f} MB)")

## Understanding the JSON Output Format

The exported `poses.json` contains interactive human-human motion data:

### Metadata
- Video/sequence information
- **Coordinate system**: Camera space (meters)
- Format version: **1.0**
- SMPL-X parameters included
- Interactive pose timing

### Per-Frame Data
For each frame, you get:
- **Interactive pose parameters**:
  - `person_id`: Unique identifier for each person
  - `smplx_parameters`:
    - `betas`: 10D shape parameters
    - `root_orient`: 3D global orientation
    - `body_pose`: 21x3 rotation vectors
    - `translation`: 3D position
    - `gender`: Person's gender (0=male, 1=female, 2=neutral)

### Example: Load and Analyze Motion

```python
import json
import numpy as np

# Load poses
with open('outputs/Couple_6806/poses.json', 'r') as f:
    data = json.load(f)

# Get metadata
metadata = data['metadata']
print(f"Total frames: {metadata['total_frames']}")
print(f"Interactive frame: {metadata['inter_time_idx']}")

# Get frame data
frame_0 = data['frames']['0']
person_0 = frame_0['persons'][0]

print(f"Person 0 shape: {person_0['smplx_parameters']['betas']}")
print(f"Person 0 position: {person_0['smplx_parameters']['translation']}")
```

### Advanced Options

You can modify the generation parameters:
- `--text`: Text description for motion generation
- `--seed`: Random seed for reproducibility
- `--inter_time_idx`: Frame index for interactive pose (0-29)
- `--gender`: Gender for two persons (e.g., "male female")
- `--seq_len`: Sequence length (default: 30 frames)

---

## Citation

If you find this work useful, please cite:

```bibtex
@inproceedings{liu2025ponimator,
    title={Ponimator: Unfolding Interactive Pose for Versatile Human-Human Interaction Animation},
    author={Liu, Shaowei and Guo, Chuan and Zhou, Bing and Wang, Jian},
    booktitle={International Conference on Computer Vision (ICCV)},
    year={2025}
}
```